# quantum-cortex — the paired extension to sixteen seeds, and the horizon of the convolved control (register Rev62, declared before this run)

Two declared questions in one session, in separate directories, nothing overwritten:

1. **The horizon** (row 32's four floor seeds 1337, 7, 42, 11): the control plus a local convolution trained to 6000 and to 12 000 steps (each unit from scratch, as the Σ), into `conv6000-ckpt` and `conv12000-ckpt`. Reading: how many of the four take off at each horizon, per seed. At any horizon the answer is *not at this horizon*, never *never*.
2. **The paired extension**: eight new seeds (100 to 800) at 3000 steps, 8 pairs, for the three references (L3 pure, the control, the convolved control) into `ref3000-s16-ckpt`, then the two decisive arms (`RB-bare`, `RB-critical`) into `rb3000-s16-ckpt`; then the readouts over the sixteen seeds (`LATEST-rb-16seeds.json`, `READOUT-control-conv-3000steps-16seeds.json`), paired on seed at 95 % (df 15, t_crit 2.131). The eight seed rows (30 to 32) stand; the sixteen seed numbers are new rows.

**GPU T4, Internet On, about 6 hours** (references about 2 h, arms about 3 h, horizon about 1 h). **Secrets:** `GITHUB_TOKEN`; `KAGGLE_USERNAME` and `KAGGLE_KEY` for the relay (now visible in the log: every push prints a line, a failed pull prints why). Cells: 1 this header; 2 the clone; 3 the relay; 4 the resume lookup; 5 the horizon; 6 the references on the new seeds; 7 the arms on the new seeds; 8 the sixteen seed readouts; 9 the files to commit. Run 2, 3 and 4 alone first and paste their outputs; then Save & Run All. **Download the log as a file before reading any number.**


In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
!rm -rf quantum-cortex
!git clone https://{token}@github.com/vaneeckhoutnicolas/quantum-cortex.git
%cd quantum-cortex
!pip -q install numpy jsonschema


In [ ]:
import os
from kaggle_secrets import UserSecretsClient
try:
    os.environ['KAGGLE_USERNAME'] = UserSecretsClient().get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = UserSecretsClient().get_secret('KAGGLE_KEY')
    from cortex_data import relay
    print('relay:', 'on' if relay.available() else 'off (kaggle command missing)')
except Exception:
    print('relay: off (secrets KAGGLE_USERNAME / KAGGLE_KEY not attached)')


In [ ]:
# previous Outputs of this notebook mounted as Inputs (resume after a relaunch); the committed references are in the clone
import glob
PREV = {name: sorted(set(glob.glob(f'/kaggle/input/**/{name}', recursive=True)))
        for name in ('conv6000-ckpt', 'conv12000-ckpt', 'ref3000-s16-ckpt', 'rb3000-s16-ckpt')}
for k, v in PREV.items():
    print(f'resume {k} from:', v or '(none — fresh start)')
def resume(name):
    return ' '.join(f'--resume-from {p}' for p in PREV[name])


In [ ]:
# 1. the horizon: the four floor seeds of row 32 at 6000 and at 12 000 steps (each unit from scratch, as the Sigma)
!python -m cortex_eval.resumable_ladder --paths ARCH-none+local-conv --steps 6000 --seeds 1337 7 42 11 --tiers 8x128 --ckpt-dir metrics/mqar/conv6000-ckpt --time-budget-min 600 --relay quantum-cortex-relay-conv6000 {resume('conv6000-ckpt')}
!python -m cortex_eval.resumable_ladder --paths ARCH-none+local-conv --steps 12000 --seeds 1337 7 42 11 --tiers 8x128 --ckpt-dir metrics/mqar/conv12000-ckpt --time-budget-min 600 --relay quantum-cortex-relay-conv12000 {resume('conv12000-ckpt')}


In [ ]:
# 2a. the references on the eight new seeds: L3 pure, the control, the convolved control, at the references' steps, tier and evaluation
!python -m cortex_eval.resumable_ladder --paths L3-+local-conv ARCH-none ARCH-none+local-conv --steps 3000 --seeds 100 200 300 400 500 600 700 800 --tiers 8x128 --ckpt-dir metrics/mqar/ref3000-s16-ckpt --time-budget-min 600 --relay quantum-cortex-relay-ref3000-s16 {resume('ref3000-s16-ckpt')}


In [ ]:
# 2b. the two decisive arms on the eight new seeds (their aggregate over these eight, LATEST-rb.json, in the new directory)
!python -m cortex_eval.resumable_rb --arms RB-bare RB-critical --steps 3000 --seeds 100 200 300 400 500 600 700 800 --tiers 8x128 --ckpt-dir metrics/mqar/rb3000-s16-ckpt --reference-from metrics/mqar/ref3000-s16-ckpt --time-budget-min 600 --relay quantum-cortex-relay-rb3000-s16 {resume('rb3000-s16-ckpt')}


In [ ]:
# 2c. the sixteen seed readouts: nothing trained here; the union of the committed eight and the new eight, paired on seed
!python -m cortex_eval.resumable_rb --arms RB-bare RB-critical --steps 3000 --seeds 1337 2024 7 42 99 3 11 2026 100 200 300 400 500 600 700 800 --tiers 8x128 --ckpt-dir metrics/mqar/rb3000-s16-ckpt --resume-from metrics/mqar/rb3000-ckpt --reference-from metrics/mqar/ref3000-ckpt metrics/mqar/sigma-l3-3000-ckpt metrics/mqar/ref3000-s16-ckpt --latest-name LATEST-rb-16seeds.json
!python -m cortex_eval.resumable_rb --control-conv-readout metrics/mqar/ref3000-s16-ckpt/READOUT-control-conv-3000steps-16seeds.json --ckpt-dir metrics/mqar/rb3000-s16-ckpt --resume-from metrics/mqar/rb3000-ckpt --reference-from metrics/mqar/ref3000-ckpt metrics/mqar/sigma-l3-3000-ckpt metrics/mqar/ref3000-s16-ckpt --arms RB-bare RB-critical --seeds 1337 2024 7 42 99 3 11 2026 100 200 300 400 500 600 700 800 --tiers 8x128


In [ ]:
import glob
for d in ('conv6000-ckpt', 'conv12000-ckpt', 'ref3000-s16-ckpt', 'rb3000-s16-ckpt'):
    files = sorted(glob.glob(f'metrics/mqar/{d}/*.json'))
    print(d, len(files), 'files')
print('Commit the four directories under metrics/mqar/ — the units are the record.')
